In [0]:
from pyspark.sql.functions import col, max

silver_orders_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/orders"
)

silver_orders_df = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

print("Current Silver Orders:", silver_orders_df.count())

latest_updated_date = (
    silver_orders_df
    .select(max("updated_date").alias("latest_updated_date"))
    .collect()[0]["latest_updated_date"]
)

print("Latest updated_date:", latest_updated_date)

Current Silver Orders: 500
Latest updated_date: 2026-01-02 16:00:00


In [0]:
from pyspark.sql.functions import to_timestamp

incremental_orders = [
    (
        1,
        1,
        101,
        "2026-10-08 10:00:00",
        "Shipped",
        "Online",
        1,
        "2026-10-08 10:00:00"
    ),
    (
        501,
        25,
        105,
        "2026-10-08 11:00:00",
        "Confirmed",
        "Store",
        501,
        "2026-10-08 11:00:00"
    )
]

columns = [
    "order_id",
    "customer_id",
    "store_id",
    "order_date",
    "order_status",
    "sales_channel",
    "payment_id",
    "updated_date"
]

incremental_orders_df = (
    spark.createDataFrame(
        incremental_orders,
        columns
    )
    .withColumn(
        "order_date",
        to_timestamp("order_date")
    )
    .withColumn(
        "updated_date",
        to_timestamp("updated_date")
    )
)

print("Incoming incremental records:", incremental_orders_df.count())

display(incremental_orders_df)

Incoming incremental records: 2


order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date
1,1,101,2026-10-08T10:00:00.000Z,Shipped,Online,1,2026-10-08T10:00:00.000Z
501,25,105,2026-10-08T11:00:00.000Z,Confirmed,Store,501,2026-10-08T11:00:00.000Z


In [0]:
from delta.tables import DeltaTable

silver_orders_delta = DeltaTable.forPath(
    spark,
    silver_orders_path
)

(
    silver_orders_delta.alias("target")
    .merge(
        incremental_orders_df.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Incremental MERGE completed successfully.")

Incremental MERGE completed successfully.


In [0]:
from pyspark.sql.functions import col

# Reload Silver Orders after MERGE
silver_orders_after_df = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

print("Silver Orders after MERGE:", silver_orders_after_df.count())

# Check updated existing order
print("Updated Order:")
display(
    silver_orders_after_df
    .filter(col("order_id") == 1)
)

# Check newly inserted order
print("New Order:")
display(
    silver_orders_after_df
    .filter(col("order_id") == 501)
)

Silver Orders after MERGE: 502
Updated Order:


order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date
1,1,101,2026-10-08T10:00:00.000Z,Shipped,Online,1,2026-10-08T10:00:00.000Z


New Order:


order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date
501,25,105,2026-10-08T11:00:00.000Z,Confirmed,Store,501,2026-10-08T11:00:00.000Z


In [0]:
latest_watermark = (
    silver_orders_after_df
    .select("updated_date")
    .agg({"updated_date": "max"})
    .collect()[0][0]
)

print("Latest watermark:", latest_watermark)

Latest watermark: 2026-10-08 11:00:00


In [0]:
watermark_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "control/watermarks"
)

In [0]:
from pyspark.sql.functions import to_timestamp

watermark_data = [
    (
        "orders_incremental",
        latest_watermark
    )
]

watermark_columns = [
    "pipeline_name",
    "last_watermark"
]

watermark_df = (
    spark.createDataFrame(
        watermark_data,
        watermark_columns
    )
)

display(watermark_df)

pipeline_name,last_watermark
orders_incremental,2026-10-08T11:00:00.000Z


In [0]:
(
    watermark_df.write
    .format("delta")
    .mode("overwrite")
    .save(watermark_path)
)

print("Watermark table created successfully.")

Watermark table created successfully.


In [0]:
watermark_df = (
    spark.read
    .format("delta")
    .load(watermark_path)
)

current_watermark = (
    watermark_df
    .filter(
        col("pipeline_name") == "orders_incremental"
    )
    .select("last_watermark")
    .collect()[0]["last_watermark"]
)

print("Current watermark:", current_watermark)

Current watermark: 2026-10-08 11:00:00


In [0]:
incremental_source_df = incremental_orders_df.filter(
    col("updated_date") > current_watermark
)

print(
    "Records after watermark:",
    incremental_source_df.count()
)

display(incremental_source_df)

Records after watermark: 0


order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date


In [0]:
from pyspark.sql.functions import to_timestamp

second_incremental_orders = [
    (
        1,
        1,
        101,
        "2026-10-08 10:00:00",
        "Delivered",
        "Online",
        1,
        "2026-10-09 09:00:00"
    ),
    (
        502,
        30,
        108,
        "2026-10-09 09:30:00",
        "Confirmed",
        "Online",
        502,
        "2026-10-09 09:30:00"
    ),
    (
        503,
        45,
        110,
        "2026-10-09 10:00:00",
        "Confirmed",
        "Store",
        503,
        "2026-10-09 10:00:00"
    )
]

columns = [
    "order_id",
    "customer_id",
    "store_id",
    "order_date",
    "order_status",
    "sales_channel",
    "payment_id",
    "updated_date"
]

second_incremental_df = (
    spark.createDataFrame(
        second_incremental_orders,
        columns
    )
    .withColumn("order_date", to_timestamp("order_date"))
    .withColumn("updated_date", to_timestamp("updated_date"))
)

print(
    "Second source batch:",
    second_incremental_df.count()
)

display(second_incremental_df)

Second source batch: 3


order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date
1,1,101,2026-10-08T10:00:00.000Z,Delivered,Online,1,2026-10-09T09:00:00.000Z
502,30,108,2026-10-09T09:30:00.000Z,Confirmed,Online,502,2026-10-09T09:30:00.000Z
503,45,110,2026-10-09T10:00:00.000Z,Confirmed,Store,503,2026-10-09T10:00:00.000Z


In [0]:
second_incremental_filtered_df = (
    second_incremental_df
    .filter(
        col("updated_date") > current_watermark
    )
)

print(
    "Records after watermark:",
    second_incremental_filtered_df.count()
)

display(second_incremental_filtered_df)

Records after watermark: 3


order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date
1,1,101,2026-10-08T10:00:00.000Z,Delivered,Online,1,2026-10-09T09:00:00.000Z
502,30,108,2026-10-09T09:30:00.000Z,Confirmed,Online,502,2026-10-09T09:30:00.000Z
503,45,110,2026-10-09T10:00:00.000Z,Confirmed,Store,503,2026-10-09T10:00:00.000Z


In [0]:
from delta.tables import DeltaTable

silver_orders_delta = DeltaTable.forPath(
    spark,
    silver_orders_path
)

(
    silver_orders_delta.alias("target")
    .merge(
        second_incremental_filtered_df.alias("source"),
        "target.order_id = source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Second incremental MERGE completed successfully.")

Second incremental MERGE completed successfully.


In [0]:
silver_orders_after_second_merge = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

print(
    "Silver Orders after second MERGE:",
    silver_orders_after_second_merge.count()
)

display(
    silver_orders_after_second_merge
    .filter(
        col("order_id").isin(1, 502, 503)
    )
    .orderBy("order_id")
)

Silver Orders after second MERGE: 504


order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date
1,1,101,2026-10-08T10:00:00.000Z,Delivered,Online,1,2026-10-09T09:00:00.000Z
502,30,108,2026-10-09T09:30:00.000Z,Confirmed,Online,502,2026-10-09T09:30:00.000Z
503,45,110,2026-10-09T10:00:00.000Z,Confirmed,Store,503,2026-10-09T10:00:00.000Z


In [0]:
new_watermark = (
    second_incremental_filtered_df
    .select("updated_date")
    .agg({"updated_date": "max"})
    .collect()[0][0]
)

print("New watermark:", new_watermark)

New watermark: 2026-10-09 10:00:00


In [0]:
from pyspark.sql import Row

new_watermark_df = spark.createDataFrame(
    [
        Row(
            pipeline_name="orders_incremental",
            last_watermark=new_watermark
        )
    ]
)

watermark_delta = DeltaTable.forPath(
    spark,
    watermark_path
)

(
    watermark_delta.alias("target")
    .merge(
        new_watermark_df.alias("source"),
        "target.pipeline_name = source.pipeline_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Watermark updated successfully.")

Watermark updated successfully.


In [0]:
updated_watermark_df = (
    spark.read
    .format("delta")
    .load(watermark_path)
)

display(updated_watermark_df)

pipeline_name,last_watermark
orders_incremental,2026-10-09T10:00:00.000Z


In [0]:
from pyspark.sql.functions import col

updated_watermark_df = (
    spark.read
    .format("delta")
    .load(watermark_path)
)

current_watermark = (
    updated_watermark_df
    .filter(col("pipeline_name") == "orders_incremental")
    .select("last_watermark")
    .collect()[0]["last_watermark"]
)

print("Current watermark:", current_watermark)

Current watermark: 2026-10-09 10:00:00


In [0]:
retry_batch_df = second_incremental_df

retry_incremental_df = (
    retry_batch_df
    .filter(
        col("updated_date") > current_watermark
    )
)

print(
    "Records after watermark:",
    retry_incremental_df.count()
)

display(retry_incremental_df)

Records after watermark: 0


order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date


In [0]:
final_orders_df = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)

print("Final Silver Orders:", final_orders_df.count())

Final Silver Orders: 504


In [0]:
display(
    final_orders_df
    .filter(
        col("order_id").isin(1, 501, 502, 503)
    )
    .select(
        "order_id",
        "order_status",
        "updated_date"
    )
    .orderBy("order_id")
)

order_id,order_status,updated_date
1,Delivered,2026-10-09T09:00:00.000Z
501,Confirmed,2026-10-08T11:00:00.000Z
502,Confirmed,2026-10-09T09:30:00.000Z
503,Confirmed,2026-10-09T10:00:00.000Z
